# PyTorch Training and ONNX Export Tutorial

This notebook demonstrates how to train PyTorch models and export them to ONNX format.

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np

from src.models.neural_architectures import (
    TransformerEncoder,
    MemoryNetwork,
    ReasoningNetwork,
    PolicyNetwork,
    WorldModel
)
from src.models.pytorch_trainer import BaseTrainer
from src.models.onnx_utils import ONNXExporter, ONNXInferenceEngine, ModelConverter

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. Training a Reasoning Network

In [ ]:
# Create model
model = ReasoningNetwork(
    input_dim=64,
    hidden_dim=128,
    output_dim=32,
    num_reasoning_steps=3
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [ ]:
# Create dataset
num_samples = 1000
X = torch.randn(num_samples, 64)
y = torch.randn(num_samples, 32)
dataset = TensorDataset(X, y)

# Split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# Setup training
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

trainer = BaseTrainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    gradient_clip=1.0
)

# Train
trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    save_dir="../../data/models/notebook_trained",
    early_stopping_patience=5
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(trainer.history['train_loss'], label='Train Loss')
plt.plot(trainer.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(trainer.history['val_loss'])
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Validation Loss Over Time')
plt.grid(True)

plt.tight_layout()
plt.show()

## 2. Export to ONNX

In [ ]:
# Export model to ONNX
model.eval()

dummy_input = torch.randn(1, 64)

onnx_path = ModelConverter.pytorch_to_onnx(
    model=model,
    model_name="reasoning_network_notebook",
    dummy_input=dummy_input,
    output_dir="../../data/models/onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
)

print(f"Model exported to: {onnx_path}")

## 3. ONNX Inference

In [ ]:
# Load ONNX model
engine = ONNXInferenceEngine(onnx_path)

# Get model info
info = engine.get_model_info()
print("Model Info:")
print(f"  Inputs: {info['input_names']}")
print(f"  Outputs: {info['output_names']}")

In [ ]:
# Compare PyTorch vs ONNX output
test_input = torch.randn(5, 64)

# PyTorch inference
with torch.no_grad():
    pytorch_output = model(test_input)

# ONNX inference
onnx_output = engine.infer(test_input.numpy())

# Compare
diff = np.abs(pytorch_output.numpy() - onnx_output)
max_diff = diff.max()
mean_diff = diff.mean()

print(f"Max difference: {max_diff:.6f}")
print(f"Mean difference: {mean_diff:.6f}")
print(f"\nOutputs match: {max_diff < 1e-5}")

## 4. Benchmark Inference Speed

In [ ]:
import time

# Prepare test data
test_data = np.random.randn(1, 64).astype(np.float32)
num_runs = 1000

# Warmup
for _ in range(10):
    _ = engine.infer(test_data)

# Benchmark ONNX
start = time.time()
for _ in range(num_runs):
    _ = engine.infer(test_data)
onnx_time = (time.time() - start) / num_runs

# Benchmark PyTorch
test_tensor = torch.from_numpy(test_data)
for _ in range(10):
    with torch.no_grad():
        _ = model(test_tensor)

start = time.time()
for _ in range(num_runs):
    with torch.no_grad():
        _ = model(test_tensor)
pytorch_time = (time.time() - start) / num_runs

print(f"PyTorch inference: {pytorch_time*1000:.3f}ms")
print(f"ONNX inference: {onnx_time*1000:.3f}ms")
print(f"Speedup: {pytorch_time/onnx_time:.2f}x")

## 5. Training a Memory Network

In [ ]:
# Create memory network
memory_model = MemoryNetwork(
    input_dim=32,
    hidden_dim=64,
    memory_size=50,
    num_heads=4
)

print(f"Memory Network parameters: {sum(p.numel() for p in memory_model.parameters()):,}")

# Test forward pass
test_input = torch.randn(2, 10, 32)
output, attention = memory_model(test_input)

print(f"\nInput shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention shape: {attention.shape}")

In [ ]:
# Visualize attention weights
plt.figure(figsize=(10, 6))
plt.imshow(attention[0].detach().numpy(), aspect='auto', cmap='viridis')
plt.colorbar(label='Attention Weight')
plt.xlabel('Memory Slot')
plt.ylabel('Query Position')
plt.title('Memory Attention Weights')
plt.show()

## 6. Training a Policy Network for RL

In [ ]:
# Create policy network
policy = PolicyNetwork(
    observation_dim=16,
    action_dim=4,
    hidden_dim=64,
    continuous=False
)

print(f"Policy parameters: {sum(p.numel() for p in policy.parameters()):,}")

# Test policy
obs = torch.randn(10, 16)
action_logits = policy(obs)

# Get action probabilities
action_probs = torch.softmax(action_logits, dim=-1)

print(f"\nAction probabilities shape: {action_probs.shape}")
print(f"Example action probs:\n{action_probs[0]}")

# Sample actions
dist = torch.distributions.Categorical(action_probs)
actions = dist.sample()
print(f"\nSampled actions: {actions}")

In [ ]:
# Export policy to ONNX
policy.eval()

policy_onnx_path = ModelConverter.pytorch_to_onnx(
    model=policy,
    model_name="policy_network_notebook",
    dummy_input=torch.randn(1, 16),
    output_dir="../../data/models/onnx",
    input_names=["observation"],
    output_names=["action_logits"],
    dynamic_axes={"observation": {0: "batch"}, "action_logits": {0: "batch"}}
)

print(f"Policy exported to: {policy_onnx_path}")

## 7. World Model Training

In [ ]:
# Create world model
world_model = WorldModel(
    observation_dim=16,
    action_dim=4,
    hidden_dim=64,
    latent_dim=32
)

print(f"World Model parameters: {sum(p.numel() for p in world_model.parameters()):,}")

# Test world model
obs = torch.randn(5, 16)
action = torch.randn(5, 4)

next_obs_pred, reward_pred, (mean, logvar, next_mean, next_logvar) = world_model(obs, action)

print(f"\nObservation shape: {obs.shape}")
print(f"Action shape: {action.shape}")
print(f"Predicted next obs shape: {next_obs_pred.shape}")
print(f"Predicted reward shape: {reward_pred.shape}")

## Summary

This notebook demonstrated:

1. Training PyTorch models with the BaseTrainer
2. Exporting models to ONNX format
3. Running ONNX inference
4. Comparing PyTorch vs ONNX outputs
5. Benchmarking inference speed
6. Working with different architectures (Reasoning, Memory, Policy, World Model)

Key benefits of ONNX:
- Faster inference
- Platform independence
- Deployment flexibility
- Production-ready format